## Gmail API (OAuth) — fetch your emails (tutorial)

This notebook shows how to connect to **Gmail** using **OAuth 2.0** and fetch your recent emails via the **Gmail API**.

### Before you run

- Create a Google Cloud project
- Enable **Gmail API**
- Configure **OAuth consent screen**
- Create an **OAuth Client ID** (Desktop app is simplest for notebooks)
- Download the client secrets JSON and save it as `credentials.json` next to this notebook

### Files created/used

- `.env` (already created): points to `credentials.json`, `token.json`, and scopes
- `credentials.json`: OAuth client secrets (you download this)
- `token.json`: cached OAuth tokens (created by this notebook after login)

`credentials.json`, `token.json`, and `.env` should stay **local** (they’re gitignored).

In [140]:
# If you're in a fresh environment, install dependencies:
# (You can also: pip install -r requirements.txt)

%pip -q install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()

CLIENT_SECRETS_FILE = os.getenv("GOOGLE_CLIENT_SECRETS_FILE", "credentials.json")
TOKEN_FILE = os.getenv("GMAIL_TOKEN_FILE", "token.json")
SCOPES = os.getenv("GMAIL_SCOPES", "https://www.googleapis.com/auth/gmail.readonly").split()

print("CLIENT_SECRETS_FILE:", CLIENT_SECRETS_FILE)
print("TOKEN_FILE:", TOKEN_FILE)
print("SCOPES:", SCOPES)

if not Path(CLIENT_SECRETS_FILE).exists():
    raise FileNotFoundError(
        f"Missing {CLIENT_SECRETS_FILE}. Download OAuth client secrets JSON from Google Cloud Console "
        f"and save it next to this notebook (or update GOOGLE_CLIENT_SECRETS_FILE in .env)."
    )

CLIENT_SECRETS_FILE: credentials.json
TOKEN_FILE: token.json
SCOPES: ['https://www.googleapis.com/auth/gmail.readonly']


In [2]:
def get_gmail_service(*, client_secrets_file: str, token_file: str, scopes: list[str]):
    """Returns an authenticated Gmail API service. Creates/refreshes token_file as needed."""
    creds = None
    token_path = Path(token_file)

    if token_path.exists():
        creds = Credentials.from_authorized_user_file(token_file, scopes)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(client_secrets_file, scopes)
            # This opens a browser window to let you sign in and approve.
            # port=0 picks an available local port automatically.
            creds = flow.run_local_server(port=0)

        token_path.write_text(creds.to_json(), encoding="utf-8")

    return build("gmail", "v1", credentials=creds)


service = get_gmail_service(
    client_secrets_file=CLIENT_SECRETS_FILE,
    token_file=TOKEN_FILE,
    scopes=SCOPES,
)

print("Gmail service created.")

Gmail service created.


In [3]:
def _header(headers: list[dict], name: str) -> str | None:
    name_lower = name.lower()
    for h in headers or []:
        if (h.get("name") or "").lower() == name_lower:
            return h.get("value")
    return None


def fetch_recent_messages(max_results: int = 5, query: str | None = None, verbose: bool = False):
    """Fetches recent messages and prints basic metadata, supports pagination for >500 results."""
    try:
        all_msgs = []
        next_page_token = None

        # Gmail maxResults per page: up to 500. If you want more, must use pagination.
        while len(all_msgs) < max_results:
            remaining = max_results - len(all_msgs)
            fetch_count = min(500, remaining)
            req = service.users().messages().list(
                userId="me",
                maxResults=fetch_count,
                q=query,
                pageToken=next_page_token,
            )
            resp = req.execute()
            msgs = resp.get("messages", [])
            all_msgs.extend(msgs)

            next_page_token = resp.get("nextPageToken")
            if not next_page_token or len(all_msgs) >= max_results:
                break

        if not all_msgs:
            print("No messages found.")
            return
        if verbose:
            for i, m in enumerate(all_msgs[:max_results], start=1):
                msg = service.users().messages().get(
                    userId="me",
                    id=m["id"],
                    format="metadata",
                    metadataHeaders=["From", "To", "Subject", "Date"],
                ).execute()

                payload = msg.get("payload", {})
                headers = payload.get("headers", [])

                frm = _header(headers, "From")
                subj = _header(headers, "Subject")
                date = _header(headers, "Date")
                snippet = (msg.get("snippet") or "").replace("\n", " ")

                print(f"\n{i}. {subj or '(no subject)'}")
                print(f"   From: {frm}")
                print(f"   Date: {date}")
                print(f"   Snippet: {snippet}")

    except HttpError as e:
        # Common causes: wrong OAuth client type, wrong redirect URIs, missing Gmail API enablement,
        # or scope mismatch.
        raise RuntimeError(f"Gmail API error: {e}") from e
    return all_msgs

# Fetch the 5 most recent emails in your inbox
_ = fetch_recent_messages(max_results=3, query="in:inbox (category:primary OR category:updates)")

In [4]:
import re
import base64
import sqlite3
from datetime import datetime
from typing import Any, Optional

from bs4 import BeautifulSoup
from dateutil import parser as date_parser

from openai import OpenAI
from dotenv import load_dotenv
import os

# Load environment variables from .env file if present
load_dotenv()

# Accept either env var name
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or os.getenv("openai_api_key")
if not OPENAI_API_KEY:
    raise ValueError(
        "Missing OpenAI API key. Add OPENAI_API_KEY=... to your .env (or openai_api_key=...)."
    )

client = OpenAI(api_key=OPENAI_API_KEY)

DB_PATH = os.getenv("JOBTRACKER_DB", "jobtracker.sqlite3")
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")

print("DB_PATH:", DB_PATH)
print("MODEL:", MODEL)

DB_PATH: jobtracker.sqlite3
MODEL: gpt-5-nano


In [5]:
def init_db(db_path: str = DB_PATH) -> None:
    conn = sqlite3.connect(db_path)
    try:
        conn.execute("PRAGMA journal_mode=WAL;")
        conn.execute("PRAGMA foreign_keys=ON;")

        # Stores raw email content for the job-related subset we care about.
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS emails (
              gmail_message_id TEXT PRIMARY KEY,
              gmail_thread_id TEXT,
              internal_date_ms INTEGER,
              from_addr TEXT,
              to_addr TEXT,
              subject TEXT,
              date_raw TEXT,
              snippet TEXT,
              body_text TEXT,
              created_at TEXT NOT NULL
            );
            """
        )

        # Records that a message_id has been processed so reruns don't re-call OpenAI.
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS processed_messages (
              gmail_message_id TEXT PRIMARY KEY,
              category TEXT,
              confidence REAL,
              model TEXT,
              raw_json TEXT,
              processed_at TEXT NOT NULL
            );
            """
        )

        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS applications (
              id INTEGER PRIMARY KEY AUTOINCREMENT,
              app_key TEXT NOT NULL UNIQUE,
              company TEXT,
              job_title TEXT,
              job_id TEXT,
              source TEXT,
              applied_date TEXT,
              status TEXT NOT NULL,
              last_update_date TEXT,
              last_email_message_id TEXT,
              last_email_date_raw TEXT,
              confidence REAL,
              notes TEXT,
              created_at TEXT NOT NULL,
              updated_at TEXT NOT NULL
            );
            """
        )

        # Event history (email-derived + manual updates)
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS app_events (
              id INTEGER PRIMARY KEY AUTOINCREMENT,
              app_key TEXT NOT NULL,
              event_type TEXT NOT NULL,
              event_date TEXT,
              gmail_message_id TEXT,
              raw_json TEXT,
              created_at TEXT NOT NULL,
              FOREIGN KEY(app_key) REFERENCES applications(app_key)
            );
            """
        )

        conn.commit()
    finally:
        conn.close()


init_db()
print("DB initialized.")

DB initialized.


In [6]:
def _b64url_decode(data: str) -> bytes:
    # Gmail uses base64url ("-" and "_")
    return base64.urlsafe_b64decode(data.encode("utf-8"))


def _extract_bodies(payload: dict) -> dict[str, str]:
    """Return {'text/plain': ..., 'text/html': ...} where available."""
    out: dict[str, str] = {}

    def walk(part: dict):
        mime = part.get("mimeType")
        body = part.get("body") or {}
        data = body.get("data")
        if data and mime in ("text/plain", "text/html"):
            try:
                out[mime] = _b64url_decode(data).decode("utf-8", errors="replace")
            except Exception:
                out[mime] = ""

        for p in part.get("parts") or []:
            walk(p)

    walk(payload or {})
    return out


def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    # Remove script/style
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text("\n")
    # Normalize whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def get_message_full(message_id: str) -> dict:
    return service.users().messages().get(userId="me", id=message_id, format="full").execute()


def normalize_date(date_raw: Optional[str]) -> Optional[str]:
    if not date_raw:
        return None
    try:
        dt = date_parser.parse(date_raw)
        return dt.isoformat()
    except Exception:
        return None

In [7]:
JOB_EMAIL_SYSTEM_PROMPT = """You are a precise information extraction system.

You will be given a single email about jobs/careers (or not related).
Your task: decide whether it relates to a job application the user made, and if so, classify the event and extract key fields.

Return ONLY valid JSON matching this schema:
{
  "is_job_related": boolean,
  "category": "application_confirmation" | "rejection" | "follow_up" | "interview" | "offer" | "job_alert" | "newsletter" | "other",
  "company": string | null,
  "job_title": string | null,
  "job_id": string | null,
  "applied_date": string | null,        // ISO 8601 if present
  "event_date": string | null,          // ISO 8601 if present (email date is acceptable if nothing else)
  "confidence": number,                // 0..1
  "reason": string,                    // short explanation
  "evidence": {
    "company": string | null,
    "job_title": string | null,
    "job_id": string | null
  },
  "notes": string | null
}

Rules:
- If it is not about a job application process (e.g. grocery promos, receipts), set is_job_related=false.
- If it's about a job posting alert or LinkedIn “add connection” etc, keep is_job_related=true but category="job_alert" or "other".
- Prefer company/job_title/job_id only when clearly supported; otherwise null.
"""


def classify_email_with_openai(*, subject: str | None, from_addr: str | None, date_raw: str | None, snippet: str | None, body_text: str | None) -> dict[str, Any]:
    user_content = {
        "from": from_addr,
        "subject": subject,
        "date": date_raw,
        "snippet": snippet,
        "body": (body_text or "")[:1500],
    }

    resp = client.chat.completions.create(
        model=MODEL,
        # temperature=0,
        messages=[
            {"role": "system", "content": JOB_EMAIL_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(user_content, ensure_ascii=False)},
        ],
        response_format={"type": "json_object"},
    )

    content = resp.choices[0].message.content or "{}"
    return json.loads(content)

In [8]:
ALLOWED_APP_CATEGORIES = {"application_confirmation", "interview", "rejection"}


def make_app_key(company: Optional[str], job_title: Optional[str], job_id: Optional[str]) -> str:
    # Stable-ish unique key; if job_id exists use it, otherwise fall back to company+title.
    c = (company or "").strip().lower()
    t = (job_title or "").strip().lower()
    j = (job_id or "").strip().lower()
    if j:
        return f"job_id:{j}"
    if c or t:
        return f"company_title:{c}|{t}".strip("|")
    # Worst case: unknown bucket
    return "unknown"


def is_message_processed(db_path: str, gmail_message_id: str) -> bool:
    conn = sqlite3.connect(db_path)
    try:
        row = conn.execute(
            "SELECT 1 FROM processed_messages WHERE gmail_message_id = ? LIMIT 1;",
            (gmail_message_id,),
        ).fetchone()
        return row is not None
    finally:
        conn.close()


def mark_message_processed(db_path: str, gmail_message_id: str, extracted: dict[str, Any]) -> None:
    now = datetime.utcnow().isoformat()
    conn = sqlite3.connect(db_path)
    try:
        conn.execute(
            """
            INSERT INTO processed_messages (gmail_message_id, category, confidence, model, raw_json, processed_at)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(gmail_message_id) DO UPDATE SET
              category=excluded.category,
              confidence=excluded.confidence,
              model=excluded.model,
              raw_json=excluded.raw_json,
              processed_at=excluded.processed_at;
            """,
            (
                gmail_message_id,
                extracted.get("category"),
                extracted.get("confidence"),
                MODEL,
                json.dumps(extracted, ensure_ascii=False),
                now,
            ),
        )
        conn.commit()
    finally:
        conn.close()


def upsert_email_and_event(
    *,
    db_path: str,
    gmail_message_id: str,
    gmail_thread_id: Optional[str],
    internal_date_ms: Optional[int],
    from_addr: Optional[str],
    to_addr: Optional[str],
    subject: Optional[str],
    date_raw: Optional[str],
    snippet: Optional[str],
    body_text: Optional[str],
    extracted: dict[str, Any],
) -> None:
    """Write to SQL ONLY for confirmation/interview/rejection categories."""
    now = datetime.utcnow().isoformat()

    status = extracted.get("category") or "other"
    if status not in ALLOWED_APP_CATEGORIES:
        return

    app_key = make_app_key(extracted.get("company"), extracted.get("job_title"), extracted.get("job_id"))
    conf = extracted.get("confidence")

    conn = sqlite3.connect(db_path)
    try:
        # Store the email (only for allowed categories)
        conn.execute(
            """
            INSERT INTO emails (
              gmail_message_id, gmail_thread_id, internal_date_ms,
              from_addr, to_addr, subject, date_raw, snippet, body_text, created_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(gmail_message_id) DO UPDATE SET
              gmail_thread_id=excluded.gmail_thread_id,
              internal_date_ms=excluded.internal_date_ms,
              from_addr=excluded.from_addr,
              to_addr=excluded.to_addr,
              subject=excluded.subject,
              date_raw=excluded.date_raw,
              snippet=excluded.snippet,
              body_text=excluded.body_text;
            """,
            (
                gmail_message_id,
                gmail_thread_id,
                internal_date_ms,
                from_addr,
                to_addr,
                subject,
                date_raw,
                snippet,
                body_text,
                now,
            ),
        )

        applied_date = extracted.get("applied_date")
        event_date = extracted.get("event_date") or normalize_date(date_raw)

        conn.execute(
            """
            INSERT INTO applications (
              app_key, company, job_title, job_id, source, applied_date,
              status, last_update_date, last_email_message_id, last_email_date_raw,
              confidence, notes, created_at, updated_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(app_key) DO UPDATE SET
              company=COALESCE(excluded.company, applications.company),
              job_title=COALESCE(excluded.job_title, applications.job_title),
              job_id=COALESCE(excluded.job_id, applications.job_id),
              status=excluded.status,
              last_update_date=excluded.last_update_date,
              last_email_message_id=excluded.last_email_message_id,
              last_email_date_raw=excluded.last_email_date_raw,
              confidence=excluded.confidence,
              notes=excluded.notes,
              updated_at=excluded.updated_at;
            """,
            (
                app_key,
                extracted.get("company"),
                extracted.get("job_title"),
                extracted.get("job_id"),
                from_addr,
                applied_date,
                status,
                event_date,
                gmail_message_id,
                date_raw,
                conf,
                extracted.get("notes") or extracted.get("reason"),
                now,
                now,
            ),
        )

        conn.execute(
            """
            INSERT INTO app_events (app_key, event_type, event_date, gmail_message_id, raw_json, created_at)
            VALUES (?, ?, ?, ?, ?, ?);
            """,
            (
                app_key,
                status,
                event_date,
                gmail_message_id,
                json.dumps(extracted, ensure_ascii=False),
                now,
            ),
        )

        conn.commit()
    finally:
        conn.close()

# Core

In [9]:
def process_inbox_to_db(max_results: int = 50, query: str = "in:inbox", *, force: bool = False):
    """Fetch messages from Gmail, classify with OpenAI, and store ONLY allowed categories.

    - Rerun-safe: skips message IDs already in processed_messages (unless force=True)
    - Writes to SQL (emails/applications/app_events) only for confirmation/interview/rejection
    """
    msgs = fetch_recent_messages(max_results=max_results, query=query)
    print(f"Found {len(msgs)} messages.")

    skipped = 0
    processed = 0
    stored = 0

    for idx, m in enumerate(msgs, start=1):
        message_id = m["id"]

        if not force and is_message_processed(DB_PATH, message_id):
            skipped += 1
            continue

        # Pull full payload so we can extract body
        full = get_message_full(message_id)
        payload = full.get("payload") or {}
        headers = payload.get("headers") or []

        from_addr = _header(headers, "From")
        to_addr = _header(headers, "To")
        subject = _header(headers, "Subject")
        date_raw = _header(headers, "Date")
        snippet = (full.get("snippet") or "").replace("\n", " ")

        bodies = _extract_bodies(payload)
        body_text = bodies.get("text/plain")
        if not body_text and bodies.get("text/html"):
            body_text = html_to_text(bodies["text/html"])

        extracted = classify_email_with_openai(
            subject=subject,
            from_addr=from_addr,
            date_raw=date_raw,
            snippet=snippet,
            body_text=body_text,
        )

        # Always mark as processed so we never re-call OpenAI for this message_id
        mark_message_processed(DB_PATH, message_id, extracted)
        processed += 1

        # Only store application records for allowed categories
        before = extracted.get("category")
        upsert_email_and_event(
            db_path=DB_PATH,
            gmail_message_id=message_id,
            gmail_thread_id=full.get("threadId"),
            internal_date_ms=int(full.get("internalDate")) if full.get("internalDate") else None,
            from_addr=from_addr,
            to_addr=to_addr,
            subject=subject,
            date_raw=date_raw,
            snippet=snippet,
            body_text=body_text,
            extracted=extracted,
        )
        if before in ALLOWED_APP_CATEGORIES:
            stored += 1

        cat = extracted.get("category")
        comp = extracted.get("company")
        title = extracted.get("job_title")
        print(f"[{idx}/{len(msgs)}] {cat} | {comp} | {title} | subj={subject!r}")

    print(f"Done. skipped={skipped} processed_now={processed} stored_app_records={stored}")


# Run this to process your recent inbox messages (rerun-safe)
process_inbox_to_db(max_results=800, query="in:inbox (category:primary OR category:updates)", force=False)

Found 800 messages.


/var/folders/45/qw19hr3d1bbb8l9_94cjt3rm0000gn/T/ipykernel_94834/910478535.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().isoformat()
/var/folders/45/qw19hr3d1bbb8l9_94cjt3rm0000gn/T/ipykernel_94834/910478535.py:73: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().isoformat()


[1/800] other | None | None | subj='Davoud just messaged you'
Done. skipped=799 processed_now=1 stored_app_records=0


In [10]:
def fetch_applications(status: str | None = None, company_like: str | None = None):
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        sql = "SELECT * FROM applications WHERE 1=1"
        params: list[Any] = []
        if status:
            sql += " AND status = ?"
            params.append(status)
        if company_like:
            sql += " AND company LIKE ?"
            params.append(f"%{company_like}%")
        sql += " ORDER BY updated_at DESC LIMIT 100"

        rows = conn.execute(sql, params).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


# Example: show the most recent 20 app records
apps = fetch_applications()
apps[:20]

[{'id': 13,
  'app_key': 'company_title:canadian tire corporation|senior data scientist',
  'company': 'Canadian Tire Corporation',
  'job_title': 'Senior Data Scientist',
  'job_id': None,
  'source': 'Jalen Wong <Jalen.Wong@cantire.com>',
  'applied_date': None,
  'status': 'interview',
  'last_update_date': '2026-05-08T17:40:20+00:00',
  'last_email_message_id': '19e08ad24bca0aa0',
  'last_email_date_raw': 'Fri, 8 May 2026 17:40:20 +0000',
  'confidence': 0.85,
  'notes': 'Second-round in-person interview details; date not provided in email.',
  'created_at': '2026-05-07T22:46:33.607779',
  'updated_at': '2026-05-13T17:05:21.317134'},
 {'id': 180,
  'app_key': 'company_title:|data scientist',
  'company': None,
  'job_title': 'Data Scientist',
  'job_id': None,
  'source': 'Indeed Apply <indeedapply@indeed.com>',
  'applied_date': '2026-04-02T18:45:20+00:00',
  'status': 'application_confirmation',
  'last_update_date': '2026-05-08T18:20:53+00:00',
  'last_email_message_id': '19e08d

In [11]:
def update_application_status(app_key: str, new_status: str, note: str | None = None):
    now = datetime.utcnow().isoformat()
    conn = sqlite3.connect(DB_PATH)
    try:
        conn.execute(
            """
            UPDATE applications
            SET status = ?, notes = COALESCE(?, notes), updated_at = ?
            WHERE app_key = ?;
            """,
            (new_status, note, now, app_key),
        )

        conn.execute(
            """
            INSERT INTO app_events (app_key, event_type, event_date, gmail_message_id, raw_json, created_at)
            VALUES (?, ?, ?, NULL, ?, ?);
            """,
            (
                app_key,
                f"manual:{new_status}",
                now,
                json.dumps({"note": note, "status": new_status}, ensure_ascii=False),
                now,
            ),
        )

        conn.commit()
    finally:
        conn.close()


# How to use:
# 1) Copy an app_key from the apps list cell above
# 2) Run something like:
# update_application_status("job_id:12345", "rejection", "Got rejection email on May 7")

In [12]:
def get_application_timeline(app_key: str):
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        rows = conn.execute(
            """
            SELECT event_type, event_date, gmail_message_id, created_at
            FROM app_events
            WHERE app_key = ?
            ORDER BY created_at ASC;
            """,
            (app_key,),
        ).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


# Example:
# timeline = get_application_timeline("job_id:12345")
# timeline

In [13]:
# Build a Pandas view of applications (applied + rejection if available)
# with a FAISS step to show semantic matching using TF-IDF vectors.

# %pip -q install faiss-cpu pandas scikit-learn

import pandas as pd
import numpy as np
import sqlite3

import faiss
from sklearn.feature_extraction.text import TfidfVectorizer

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
try:
    apps = conn.execute(
        """
        SELECT app_key, company, job_title, job_id, applied_date, status, updated_at
        FROM applications
        ORDER BY updated_at DESC;
        """
    ).fetchall()

    if not apps:
        raise RuntimeError("No rows in applications table yet. Run process_inbox_to_db() first.")

    apps_df = pd.DataFrame([dict(r) for r in apps])

    # Grab event dates for confirmations and rejections.
    events = conn.execute(
        """
        SELECT app_key, event_type, event_date
        FROM app_events
        WHERE event_type IN ('application_confirmation', 'rejection', 'interview')
        ;
        """
    ).fetchall()

finally:
    conn.close()

events_df = pd.DataFrame([dict(r) for r in events])

# applied_date: earliest application_confirmation event_date (fallback to applications.applied_date)
apply_dates = (
    events_df[events_df["event_type"] == "application_confirmation"]
    .groupby("app_key")["event_date"]
    .min()
    .rename("date_of_applied")
    .reset_index()
)

rejection_dates = (
    events_df[events_df["event_type"] == "rejection"]
    .groupby("app_key")["event_date"]
    .min()
    .rename("date_of_rejection")
    .reset_index()
)

out = apps_df.merge(apply_dates, on="app_key", how="left").merge(rejection_dates, on="app_key", how="left")

# Status rule: if a rejection exists, mark as rejection; else keep existing status.
out["status"] = np.where(out["date_of_rejection"].notna(), "rejection", out["status"])

# FAISS semantic matching step (TF-IDF vectors)
# We embed (company + job_title) and do nearest-neighbor search; then we keep the row mapping.
# This demonstrates FAISS usage, but the final applied/rejection dates come from SQL events.
texts = (out["company"].fillna("") + " " + out["job_title"].fillna("")).tolist()
vectorizer = TfidfVectorizer(max_features=4000, ngram_range=(1, 2))
X = vectorizer.fit_transform(texts).astype(np.float32)

# Convert sparse matrix to dense for FAISS (fine for small datasets; for large scale use a different approach)
X_dense = X.toarray().astype(np.float32)

index = faiss.IndexFlatL2(X_dense.shape[1])
index.add(X_dense)

# Example: find top-1 similar record for each row (distance computed in embedding space)
D, I = index.search(X_dense, 1)
# I should generally point to itself for the same row, but we keep it to show the pipeline.
out["faiss_self_match_idx"] = I[:, 0]
out["faiss_self_distance"] = D[:, 0]

# Final display columns
final_cols = [
    "company",
    "job_title",
    "date_of_applied",
    "date_of_rejection",
    "status",
]

# If date_of_applied is missing, try applications.applied_date
out["date_of_applied"] = out["date_of_applied"].fillna(out["applied_date"])

df = out[final_cols].sort_values(by="date_of_applied", ascending=False, na_position="last")

df

,company,job_title,date_of_applied,date_of_rejection,status
15,Coca-Cola Canada Bottling Limited,Data Scientist,2026-05-13T16:14:44+00:00,NaN,application_confirmation
14,Waabi,Senior / Staff Applied Scientist,2026-05-13T16:09:56+00:00,NaN,application_confirmation
12,COGNITO INC,Data Scientist/AI/ML Lead_Healthcare,2026-05-13T15:51:10+00:00,NaN,application_confirmation
11,SearchLabs,Lead AI Engineer,2026-05-13T15:50:35+00:00,NaN,application_confirmation
10,TD,Senior Machine Learning Engineer ( R_1478728 ),2026-05-13T15:33:12+00:00,NaN,application_confirmation
...,...,...,...,...,...
234,Cantire,Senior Data Scientist,NaN,NaN,interview
235,Stripe,"2025-2026 PhD Intern, Data Scientist",NaN,NaN,interview
236,"Intuition Machines, Inc.",Senior/Lead ML Applied Scientist,NaN,2026-05-06T09:06:10Z,rejection
237,Copoly.ai,Machine Learning Bioinformatics Engineer,NaN,2026-05-07T15:40:05Z,rejection


In [14]:
df_out = df.copy()
def standardize_date_column(series):
    """Parse a pandas Series of datetimes in multiple formats, return as YYYY-MM-DD strings."""
    # Try ISO8601/mixed parsing, coerce errors to NaT
    result = pd.to_datetime(series, errors="coerce", utc=True, format='mixed')
    # Fall back: if some failed, try parsing as plain %Y-%m-%d (common fallback)
    needs_fallback = result.isna() & series.notna()
    if needs_fallback.any():
        fallback = pd.to_datetime(series[needs_fallback], errors="coerce", format="%Y-%m-%d", utc=True)
        result.loc[needs_fallback] = fallback
    return result.dt.strftime("%Y-%m-%d")

df_out["date_of_applied"] = standardize_date_column(df_out["date_of_applied"])
df_out["date_of_rejection"] = standardize_date_column(df_out["date_of_rejection"])
df_out = df_out.sort_values(by="date_of_applied", ascending=True, na_position="last").reset_index(drop=True)
df_out.to_csv("applications_with_dates.csv", index=False)

In [15]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
try:
  
      # Grab event dates for confirmations and rejections.
    events = conn.execute(
        """
        SELECT *
        FROM app_events
        WHERE event_type IN ('application_confirmation', 'rejection', 'interview')
        ;
        """
    ).fetchall()

finally:
    conn.close()
events_df = pd.DataFrame([dict(r) for r in events])

In [16]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
try:
  
      # Grab event dates for confirmations and rejections.
    events = conn.execute(
        """
        SELECT *
        FROM app_events
        WHERE event_type IN ('application_confirmation', 'rejection', 'interview')
        ;
        """
    ).fetchall()

finally:
    conn.close()
events_df = pd.DataFrame([dict(r) for r in events])

In [17]:
import json

def build_app_key_from_row(row):
    """
    Given a row from a DataFrame where one column is 'raw_json' (a JSON-encoded string), 
    returns a string app_key "company | job_title | job_id", using 'None' if any field is missing.
    """
    row_dict = json.loads(row['raw_json'])
    company = row_dict.get('company') if row_dict.get('company') is not None else 'None'
    job_title = row_dict.get('job_title') if row_dict.get('job_title') is not None else 'None'
    job_id = row_dict.get('job_id') if row_dict.get('job_id') is not None else 'None'
    return f"{company} | {job_title} | {job_id}"
def build_job_title(row):
    row_dict = json.loads(row['raw_json'])
    job_title = row_dict.get('job_title') if row_dict.get('job_title') is not None else 'None'
    return job_title
def build_company(row):
    row_dict = json.loads(row['raw_json'])
    company = row_dict.get('company') if row_dict.get('company') is not None else 'None'
    return company
def build_job_id(row):
    row_dict = json.loads(row['raw_json'])
    job_id = row_dict.get('job_id') if row_dict.get('job_id') is not None else 'None'
    return job_id
events_df['app_key'] = events_df.apply(build_app_key_from_row, axis=1)
events_df['job_title'] = events_df.apply(build_job_title, axis=1)
events_df['company'] = events_df.apply(build_company, axis=1)
events_df['job_id'] = events_df.apply(build_job_id, axis=1)

In [18]:
events_df


,id,app_key,event_type,event_date,gmail_message_id,raw_json,created_at,job_title,company,job_id
0,1,"Huawei Technologies Canada Co., Ltd. | Intern ...",interview,2026-05-07T21:46:08+00:00,19e0467cc72bdc42,"{""is_job_related"": true, ""category"": ""intervie...",2026-05-07T22:23:20.553067,Intern Researcher - AI Agent,"Huawei Technologies Canada Co., Ltd.",None
1,2,"Waabi | Research Engineer, Sensor Signal Proce...",application_confirmation,2026-05-07T19:34:02+00:00,19e03ef011436be0,"{""is_job_related"": true, ""category"": ""applicat...",2026-05-07T22:24:16.530533,"Research Engineer, Sensor Signal Processing",Waabi,None
2,3,Xanadu Quantum Technologies | AI Specialist - ...,application_confirmation,2026-05-07T17:11:55Z,19e036cb6e4b83f3,"{""is_job_related"": true, ""category"": ""applicat...",2026-05-07T22:25:26.429805,AI Specialist - AI For Research,Xanadu Quantum Technologies,None
3,4,DraftAid | ML Engineer | None,rejection,2026-05-07T12:56:40-04:00,19e035f4c7ed5fcd,"{""is_job_related"": true, ""category"": ""rejectio...",2026-05-07T22:25:43.560227,ML Engineer,DraftAid,None
4,5,RBC | Staff Data/AI Engineer | R-0000169632,application_confirmation,2026-05-07T16:23:49+00:00,19e0340aade71301,"{""is_job_related"": true, ""category"": ""applicat...",2026-05-07T22:26:40.550968,Staff Data/AI Engineer,RBC,R-0000169632
...,...,...,...,...,...,...,...,...,...,...
319,321,Canadian Tire Corporation | Senior Data Scient...,interview,2026-05-11,19e0919e45e51127,"{""is_job_related"": true, ""category"": ""intervie...",2026-05-13T17:04:51.671999,Senior Data Scientist,Canadian Tire Corporation,None
320,322,Canadian Tire Corporation | Senior Data Scient...,interview,2026-05-08T19:38:13+00:00,19e091917fabc5b6,"{""is_job_related"": true, ""category"": ""intervie...",2026-05-13T17:05:21.271012,Senior Data Scientist,Canadian Tire Corporation,None
321,323,Ada | None | None,application_confirmation,2026-05-08T19:29:16+00:00,19e0910d0043a723,"{""is_job_related"": true, ""category"": ""applicat...",2026-05-13T17:05:21.291206,None,Ada,None
322,324,None | Data Scientist | None,application_confirmation,2026-05-08T18:20:53+00:00,19e08d234b4f62be,"{""is_job_related"": true, ""category"": ""applicat...",2026-05-13T17:05:21.302159,Data Scientist,None,None


In [19]:
from openai import OpenAI
import os
import numpy as np
from tqdm import tqdm

# Accept either env var name
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or os.getenv("openai_api_key")
if not OPENAI_API_KEY:
    raise ValueError(
        "Missing OpenAI API key. Add OPENAI_API_KEY=... to your .env (or openai_api_key=...)."
    )
client = OpenAI(api_key=OPENAI_API_KEY)

def get_embeddings_batch(texts, client, batch_size=64):
    """
    Efficiently get embeddings for a list of texts by batching requests.
    Returns a list of embedding vectors (length = len(texts)).
    """
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch_texts = texts[i:i+batch_size]
        response = client.embeddings.create(
            input=batch_texts,
            model="text-embedding-3-small",
        )
        # Ensure order is preserved
        embeddings.extend([r.embedding for r in response.data])
    return embeddings

# Get embeddings for all "app_key" values at once (in batches)
app_key_texts = events_df['app_key'].tolist()
app_key_embeddings = get_embeddings_batch(app_key_texts, client)

# Store embeddings into the dataframe
events_df['app_key_embedding'] = app_key_embeddings

Embedding batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embedding batches: 100%|██████████| 6/6 [00:07<00:00,  1.28s/it]


In [20]:
from sklearn.metrics.pairwise import cosine_similarity

# Convert the list of embeddings into a numpy array
embeddings_matrix = np.vstack(events_df['app_key_embedding'].values)

# Compute the cosine similarity matrix (pairwise similarity between all applications)
cosine_sim_matrix = cosine_similarity(embeddings_matrix)

# Show the cosine similarity matrix (for example, as a DataFrame for easy inspection)
import pandas as pd
cosine_sim_df = pd.DataFrame(
    cosine_sim_matrix, 
    index=events_df.index, 
    columns=events_df.index
)


In [21]:
# Create confirmation and rejection DataFrames with renamed dates, but work on copies to avoid inplace warnings
confirmation_df = events_df[events_df['event_type'] == 'application_confirmation'].copy()
confirmation_df['confirmation_date'] = confirmation_df['event_date']

rejection_df = events_df[events_df['event_type'] == 'rejection'].copy()
rejection_df['rejection_date'] = rejection_df['event_date']

# Select relevant columns before merging to keep the final DataFrame clean
confirmation_cols = ['app_key', 'confirmation_date', 'company', 'job_title', 'job_id']
rejection_cols = ['app_key', 'rejection_date']

confirmation_df = confirmation_df[confirmation_cols]
rejection_df = rejection_df[rejection_cols]

# Merge confirmation and rejection info per application (left join so we don't lose applications with no rejection yet)
confirmation_rejection_df = pd.merge(
    confirmation_df,
    rejection_df,
    on="app_key",
    how="left"
)

# Display the resulting DataFrame
confirmation_rejection_df.to_csv("confirmation_rejection_df.csv", index=False)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Convert the list of embeddings into a numpy array
confirmation_rejection_df
embeddings_matrix = np.vstack(events_df['app_key_embedding'].values)

# Compute the cosine similarity matrix (pairwise similarity between all applications)
cosine_sim_matrix = cosine_similarity(embeddings_matrix)

# Show the cosine similarity matrix (for example, as a DataFrame for easy inspection)
import pandas as pd
cosine_sim_df = pd.DataFrame(
    cosine_sim_matrix, 
    index=events_df.index, 
    columns=events_df.index
)
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# We'll walk through each row of confirmation_df, find likely rejection(s) based on cosine similarity.
# For each confirmation, show nearest non-NaN rejection candidate(s).

# We'll use events_df to find rejections with an embedding and date.
# We'll use a threshold (or top N) nearest rejections per confirmation.

confirmation_indices = confirmation_df.index.tolist()
num_confirmations = len(confirmation_indices)
cur_idx = 8

# Find all rejections with an embedding
rejection_event_mask = (events_df['event_type'] == 'rejection') & events_df['app_key_embedding'].notnull()
rejection_indices = events_df[rejection_event_mask].index.tolist()

# Helper to display current confirmation and its best-matching rejection(s)
def show_confirmation_with_rejection(cur_idx, num_nearest=3, sim_thresh=0.9):
    clear_output(wait=True)
    idx = confirmation_indices[cur_idx]
    conf_row = confirmation_df.loc[idx]

    conf_embedding = events_df.loc[idx, 'app_key_embedding']
    # Get cosine similarity between this application and all rejections
    conf_vector = np.array(conf_embedding).reshape(1, -1)
    rejections_matrix = np.vstack(events_df.loc[rejection_indices, 'app_key_embedding'].values)
    similarities = cosine_similarity(conf_vector, rejections_matrix)[0]  # shape: (num_rejections,)

    # Find top-matching rejection(s)
    top_idxs = np.argsort(similarities)[::-1]
    nearest = []
    for i in top_idxs:
        if similarities[i] >= sim_thresh:
            nearest.append((rejection_indices[i], similarities[i]))
        if len(nearest) >= num_nearest:
            break

    md = f"## Application Confirmation\n"
    md += f"**Company:** {conf_row['company']}\n\n"
    md += f"**Job Title:** {conf_row['job_title']}\n\n"
    md += f"**Job ID:** {conf_row['job_id']}\n\n"
    md += f"**Confirmation Date:** {conf_row['confirmation_date']}\n\n"
    md += f"**App Key:** {conf_row['app_key']}\n\n"

    if nearest:
        md += f"---\n### Possible Rejection(s) (cosine similarity >= {sim_thresh}):\n"
        for rej_idx, sim_val in nearest:
            rej_evt = events_df.loc[rej_idx]
            md += f"- **Rejection Date:** {rej_evt['event_date']}\n"
            md += f"  \t**Company:** {rej_evt.get('company', '')}\n"
            md += f"  \t**Job Title:** {rej_evt.get('job_title', '')}\n"
            md += f"  \t**Job ID:** {rej_evt.get('job_id', '')}\n"
            md += f"  \t**Rejection App Key:** {rej_evt['app_key']}\n"
            md += f"  \t**Cosine similarity:** {sim_val:.2f}\n\n"
    else:
        md += "\n_No likely rejections found (above similarity threshold)._"

    display(Markdown(md))

    buttons = []
    def on_next(b):
        nonlocal cur_idx
        if cur_idx < num_confirmations - 1:
            cur_idx += 1
            show_confirmation_with_rejection(cur_idx)
    def on_prev(b):
        nonlocal cur_idx
        if cur_idx > 0:
            cur_idx -= 1
            show_confirmation_with_rejection(cur_idx)

    next_btn = widgets.Button(description="Next confirmation")
    prev_btn = widgets.Button(description="Previous confirmation")
    next_btn.on_click(on_next)
    prev_btn.on_click(on_prev)
    controls = widgets.HBox([prev_btn, next_btn])
    display(controls)

    # Optional: ask user for manual annotation (was this a rejection?)
    # ... you could use a widget here to collect user input/label

show_confirmation_with_rejection(cur_idx)

## Application Confirmation
**Company:** McKesson

**Job Title:** Senior Data Scientist

**Job ID:** None

**Confirmation Date:** 2026-04-30T13:40:33Z

**App Key:** McKesson | Senior Data Scientist | None


_No likely rejections found (above similarity threshold)._